# Lecture 8: Deploy the Flask Model App with AWS Elastic Beanstalk

This lesson explains the final project path: GitHub stores the Flask app, AWS CodePipeline receives changes, and Elastic Beanstalk runs the app in a managed environment.

![AWS Elastic Beanstalk logo](https://www.next.inc/uploads/2023/12/aws_elastic_beanstalk.jpg)

Image source: [Elastic Beanstalk overview image](https://www.next.inc/ms/aws/key-aws-services.html). Deployment details are checked against the [AWS Python platform guide](https://docs.aws.amazon.com/elasticbeanstalk/latest/dg/create-deploy-python-container.html) and [AWS configuration-file guide](https://docs.aws.amazon.com/elasticbeanstalk/latest/dg/ebextensions.html).

## What Elastic Beanstalk does

Elastic Beanstalk is a managed AWS service for running a web application. You give it application code and configuration; it creates and manages the environment needed to run the app.

It is not simply one Linux machine. Under the hood AWS can manage resources such as compute, deployment versions, health checks, logs, and scaling settings. You still choose settings, monitor health, and pay for the resources used.

In [ ]:
# Cell 1: visualise the continuous-delivery path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

steps = ['Developer\npushes code', 'GitHub\nrepository', 'AWS\nCodePipeline', 'Elastic\nBeanstalk', 'Public Flask\nweb app']
colors = ['#D6EAF8', '#E8DAEF', '#FCF3CF', '#D5F5E3', '#FADBD8']
fig, ax = plt.subplots(figsize=(14, 3.4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3.4)
ax.axis('off')
for index, (label, color) in enumerate(zip(steps, colors)):
    x = 0.25 + index * 2.75
    box = FancyBboxPatch((x, 1.15), 2.05, 0.85, boxstyle='round,pad=0.08', facecolor=color, edgecolor='#444')
    ax.add_patch(box)
    ax.text(x + 1.025, 1.575, label, ha='center', va='center', fontsize=10, weight='bold')
    if index < len(steps) - 1:
        ax.annotate('', xy=(x + 2.65, 1.575), xytext=(x + 2.08, 1.575), arrowprops={'arrowstyle': '->', 'lw': 2})
ax.text(7, 0.45, 'A new GitHub push can trigger source → deploy automatically.', ha='center', fontsize=11)
ax.set_title('End-to-end deployment flow', fontsize=15, weight='bold')
plt.show()

## Before AWS: check the repository

The deployment package should contain the Flask code, HTML templates, requirements file, and trusted model artifact. Keep the entry files at the repository root, not inside an extra top-level zip folder.

Recommended structure:

    forest-fire-app/
    ├── application.py
    ├── Procfile
    ├── requirements.txt
    ├── models/ridge_pipeline.pkl
    └── templates/home.html

In [ ]:
# Cell 2: requirements.txt content for the Flask prediction app
requirements = ['Flask', 'gunicorn', 'numpy', 'pandas', 'scikit-learn']
print('\n'.join(requirements))
print('\nIn a real project, pin tested versions, for example Flask==3.1.0.')

### Why does requirements.txt matter?

Elastic Beanstalk installs Python dependencies listed in a requirements file during deployment. Include every package imported by application.py. A missing package is a common reason an environment becomes unhealthy.

In [ ]:
# Cell 3: the important application entry point
application_code = '''
from flask import Flask

application = Flask(__name__)
app = application

@application.route('/')
def index():
    return 'Forest Fire Predictor is running'
'''
print(application_code)

### Understand the entry point

The server needs to know two names: the Python file and the Flask application object. If the file is application.py and the object is application, the import target is application:application.

The transcript assigns app = application. That is fine, but the startup command must match the object name it exposes.

In [ ]:
# Cell 4: preferred startup command in a Procfile
procfile = 'web: gunicorn application:application'
print(procfile)
print('\nMeaning: start a web server, import application.py, then use the application Flask object.')

## Where does .ebextensions fit?

The transcript uses .ebextensions/python.config to provide a WSGI path. AWS supports .config files in a folder named .ebextensions for environment customization. The exact WSGI setting can differ by Elastic Beanstalk platform generation, so check the documentation for the Python platform selected in the AWS console.

For a simple modern Python app, a Procfile is a direct and readable way to declare the startup command. Do not add both approaches unless you understand how they interact.

In [ ]:
# Cell 5: display the transcript-style .ebextensions example for learning
ebextensions_example = '''
option_settings:
  aws:elasticbeanstalk:container:python:
    WSGIPath: application:application
'''
print(ebextensions_example)
print('Save only after verifying this setting for the selected Elastic Beanstalk Python platform.')

## Console steps: create the environment

1. In AWS Elastic Beanstalk, create an application and a web-server environment.
2. Choose the currently supported Python platform and the correct AWS Region.
3. Deploy a tested source bundle or start with a sample application.
4. Wait for environment health to become green before opening its URL.
5. If health is red, inspect Events and Logs first; common causes are wrong entry points, missing packages, or a bad model-file path.

These are console actions, not notebook code, because they create billable cloud resources.

In [ ]:
# Cell 6: visualise a practical deployment health check
checks = ['Environment health\nis green', 'Open app URL', 'Open prediction page', 'Submit known\nvalid inputs', 'Read Events/Logs\nif anything fails']
fig, ax = plt.subplots(figsize=(13, 3))
ax.axis('off')
for index, label in enumerate(checks):
    x = 0.25 + index * 2.55
    color = '#D5F5E3' if index < 4 else '#FADBD8'
    ax.add_patch(FancyBboxPatch((x, 0.9), 2.0, 0.8, boxstyle='round,pad=0.08', facecolor=color, edgecolor='#444'))
    ax.text(x + 1.0, 1.3, label, ha='center', va='center', fontsize=10, weight='bold')
    if index < len(checks) - 1:
        ax.annotate('', xy=(x + 2.45, 1.3), xytext=(x + 2.02, 1.3), arrowprops={'arrowstyle': '->', 'lw': 2})
ax.set_xlim(0, 13)
ax.set_ylim(0, 2.4)
ax.set_title('Verify the deployed prediction service', fontsize=14, weight='bold')
plt.show()

## Connect GitHub using AWS CodePipeline

1. Create a pipeline and choose GitHub as the source.
2. Authorize AWS to access only the intended repository.
3. Choose the deployment branch, such as main.
4. For this small Flask project, a build stage is optional when no build is needed.
5. Choose Elastic Beanstalk as the deploy provider, then select the application and environment.
6. Push a small change and confirm the pipeline source and deploy stages both succeed.

A webhook can start the pipeline after a new push. Treat the pipeline as the bridge between version-controlled code and the running environment.

## Quick revision card

1. GitHub stores the application source; Elastic Beanstalk runs it.
2. CodePipeline can automate GitHub → Elastic Beanstalk delivery.
3. requirements.txt tells the platform which Python packages to install.
4. The entry command must match the filename and Flask object name.
5. Check environment health, app URL, prediction route, and logs after deployment.
6. Do not commit secrets or AWS keys. Use environment properties or a secrets service.
7. Stop or terminate unused environments to avoid unnecessary cloud cost.

**One-line interview answer:** CodePipeline connects a GitHub source change to an Elastic Beanstalk deployment, while the application startup configuration tells the Python environment how to launch the Flask app.